# Score-based search (XGES / hill climbing / Tabu search)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/score_search.ipynb)

XGES, hill climbing, and Tabu search look for the graph with the best BIC score. XGES searches
over equivalence classes, as GES does, with deletions first. Hill climbing adds, removes, or
reverses one edge of a DAG at a time and stops when no move improves the score; Tabu search keeps
going past that point and returns the best DAG it visits. All three return a CPDAG and suit
linear-Gaussian data.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.5.0` from PyPI;
local kernels use the installed environment.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.5.0" pandas

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 2,000 samples.

In [ ]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

## Fit XGES, hill climbing, and Tabu search

`metadata["score"]` is the BIC score of the returned graph, which each search minimizes. A larger
`lambda_value` weighs the BIC penalty more and gives a sparser graph. With
`extended_search=False`, XGES skips its extended search for a faster fit, and `tabu_length` sets
how many recently visited DAGs Tabu search may not return to.

In [ ]:
results = {
    "XGES": andrey.xges(X),
    "Hill climbing": andrey.hc(X),
    "Tabu search": andrey.tabu(X),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}")
    display(andrey.viz.draw(result.structure))

## Compare with GES over ten datasets

Structural Hamming distance (SHD) counts errors against the truth's CPDAG; lower is better. The
empty graph's SHD is the number of true edges, so an estimate at or above it has not improved on
returning no edges. One dataset can favor any method, so the table covers ten seeds and their
median.

In [ ]:
methods = {
    "GES": lambda X: andrey.ges(X),
    "XGES": lambda X: andrey.xges(X),
    "Hill climbing": lambda X: andrey.hc(X),
    "Tabu search": lambda X: andrey.tabu(X),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="cpdag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

Tabu search starts with hill climbing's moves, so its score is never worse than hill
climbing's on the same data. The [0.3.0 release card](https://andrey.abel.ai/docs/benchmarks/0.3.0.html)
compares XGES with the xges package and the other two with bnlearn, on larger graphs with ten rows
per variable.

## Use another dataset

Change `seed`, or the `d` and `n` passed to `benchmarks.scm` and `sample`, and rerun.